<a href="https://colab.research.google.com/github/Alanoud-Alotaibi/sdaia-capstone-data-pipeline/blob/main/sdaia_capstone_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Team Setup

In [ ]:
from google.colab import userdata

github_token = userdata.get('GITHUB')
username = "Alanoud-Alotaibi"
repo = "sdaia-capstone-data-pipeline"

!git remote set-url origin https://{github_token}@github.com/{username}/{repo}.git

#teammate users account in github
!git config user.email "rawan1hamad@hotmail.com"
!git config user.name "Rawan1H"

!git config user.email "Reem.74sh@gmail.com"
!git config user.name "ReemAlshathri74"

print("تم تحديث رابط الرفع بنجاح!")

تم تحديث رابط الرفع بنجاح!


###Download Customer Support Tickets - CRM dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ajverse/customer-support-tickets-crm-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'customer-support-tickets-crm-dataset' dataset.
Path to dataset files: /kaggle/input/customer-support-tickets-crm-dataset


### Load Dataset into DataFrame

In [ ]:
import pandas as pd
import os

# Construct the full path to the CSV file
csv_file_name = 'enhanced_customer_support_data.csv'
csv_file_path = os.path.join(path, csv_file_name)

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(csv_file_path)

display(df.head())

,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,Elena Rodriguez,5
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,Anya Sharma,5
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,David Kim,5


In [ ]:
!git pull origin main
!git add .
!git commit -m "loading CRM dataset"
!git push origin main

From https://github.com/Alanoud-Alotaibi/sdaia-capstone-data-pipeline
 * branch            main       -> FETCH_HEAD
Already up to date.
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


### Install Dependcies

In [ ]:
# ==========================================
# 1. INGESTION, QUALITY & LINEAGE
# ==========================================
# kafka-python: Real Kafka client for the producer/consumer
# pydantic: For strict data contracts at the boundary
# great_expectations: For the pipeline quality gate[cite: 1]
# openlineage-python: To emit START/COMPLETE/FAIL events[cite: 1]
!pip install kafka-python pydantic great_expectations openlineage-python pandas kagglehub

# ==========================================
# 2. DELTA LAKEHOUSE
# ==========================================
# pyspark & delta-spark: Required to build the Bronze, Silver, and Gold layers with real MERGE upserts[cite: 1]
!pip install pyspark delta-spark

# ==========================================
# 3. RAG PIPELINE
# ==========================================
# Requires libraries for document chunking, embeddings, vector stores, and reranking[cite: 1]
# Note: Using ChromaDB/FAISS for the vector store and Sentence Transformers for the cross-encoder/embeddings
!pip install langchain langchain-community sentence-transformers chromadb faiss-cpu

# ==========================================
# 4. ORCHESTRATION
# ==========================================
# apache-airflow: Required to wire every stage together with task dependencies[cite: 1]
!pip install apache-airflow

# ==========================================
# 5. KAFKA LOCAL SERVER SETUP
# ==========================================
# Download and extract Kafka
!wget https://archive.apache.org/dist/kafka/3.6.0/kafka_2.13-3.6.0.tgz
!tar -xzf kafka_2.13-3.6.0.tgz

# Start Zookeeper as a background daemon
!./kafka_2.13-3.6.0/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.0/config/zookeeper.properties
!sleep 5 # Wait for Zookeeper to initialize

# Start Kafka Server as a background daemon
!./kafka_2.13-3.6.0/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.0/config/server.properties
!sleep 5 # Wait for Kafka to initialize

print("✅ All team libraries installed and Kafka is running in the background!")

### OpenLineage & Pydantic Data Contract

In [ ]:
import json
import uuid
import datetime
from pydantic import BaseModel, Field, ValidationError
from openlineage.client import OpenLineageClient
from openlineage.client.run import RunEvent, RunState, Run
from openlineage.client.job import Job

# 1. OpenLineage Setup
client = OpenLineageClient()
namespace = "sdaia-capstone"
job = Job(namespace=namespace, name="ingestion_and_quality_gate")
run = Run(runId=str(uuid.uuid4()))

def emit_lineage_event(state: RunState):
    event = RunEvent(
        eventType=state,
        eventTime=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        run=run,
        job=job,
        producer="colab-notebook"
    )
    client.emit(event)
    print(f"📡 OpenLineage Event Emitted: {state.value}")

# Emit START event as the pipeline begins
emit_lineage_event(RunState.START)

# 2. Pydantic Data Contract
class TicketContract(BaseModel):
    Ticket_ID: str = Field(..., description="Unique ticket ID")
    Customer_Name: str = Field(..., min_length=2)
    Customer_Email: str = Field(..., pattern=r"^\S+@\S+\.\S+$")
    Ticket_Description: str = Field(..., min_length=5)
    Priority_Level: str = Field(..., pattern="^(Low|Medium|High|Critical)$") # Corrected column name

### Kafka Producer (Ingestion & Quarantine Zone)

In [ ]:
from kafka import KafkaProducer
import pandas as pd
import random

producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

MAIN_TOPIC = 'bronze_raw_tickets'
DEAD_LETTER_TOPIC = 'quarantine_tickets'

# Introduce a bad record to prove the failure path
# We modify a valid row to have an invalid email and priority
bad_record = df.iloc[0].to_dict()
bad_record['Customer_Email'] = "invalid-email"
bad_record['Priority_Level'] = "SuperUrgent"
test_records = [df.iloc[1].to_dict(), bad_record]

print("🚀 Starting Data Ingestion at the Boundary...")

for record in test_records:
    try:
        # Enforce boundary using Pydantic
        validated = TicketContract(**record)
        producer.send(MAIN_TOPIC, validated.model_dump())
        print(f"✅ PASSED: Ticket {record.get('Ticket_ID')} sent to main pipeline.")

    except ValidationError as e:
        # Failure Path: Quarantine
        error_msg = str(e).replace('\n', ' ')
        quarantine_data = {
            "raw_payload": record,
            "rejection_reason": error_msg
        }
        producer.send(DEAD_LETTER_TOPIC, quarantine_data)
        print(f"🚫 REJECTED: Ticket {record.get('Ticket_ID')} routed to Dead-Letter Topic. Reason: {error_msg[:100]}...")

producer.flush()

### Kafka Consumer & Great Expectations (The Hard Gate)

In [ ]:
from kafka import KafkaConsumer
import great_expectations as gx

consumer = KafkaConsumer(
    'bronze_raw_tickets',
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    consumer_timeout_ms=5000, # Stop listening after 5 seconds of silence
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Collect data from Kafka
consumed_data = [msg.value for msg in consumer]
consumed_df = pd.DataFrame(consumed_data)

print(f"📥 Consumed {len(consumed_df)} valid records from Kafka.")

if not consumed_df.empty:
    # 1. Great Expectations Quality Gate
    context = gx.get_context(mode="ephemeral")
    datasource = context.data_sources.add_pandas("kafka_batch")
    data_asset = datasource.add_dataframe_asset(name="tickets_asset")
    batch_request = data_asset.build_batch_request(dataframe=consumed_df)

    expectation_suite_name = "ticket_quality_gate"
    context.add_or_update_expectation_suite(expectation_suite_name)
    validator = context.get_validator(
        batch_request=batch_request,
        expectation_suite_name=expectation_suite_name
    )

    # Define our strict rule: Priority must not be null
    validator.expect_column_values_to_not_be_null("Priority_Level")
    validation_result = validator.save_expectation_suite(discard_failed_expectations=False)
    checkpoint = context.add_or_update_checkpoint(
        name="gate_checkpoint",
        validator=validator,
    )

    results = checkpoint.run()

    # 2. Halt Pipeline on Failure / Lineage Emitting
    if not results.success:
        emit_lineage_event(RunState.FAIL)
        raise RuntimeError("🛑 QUALITY GATE FAILED: Pipeline halted due to Great Expectations mismatch.")
    else:
        emit_lineage_event(RunState.COMPLETE)
        print("🟢 QUALITY GATE PASSED: Data is ready for the Delta Lakehouse Bronze layer.")